# PAR_UPAR — True Resume + Official UPAR 2027 Code Submission

Bản này sửa hai lỗi quan trọng:

**True resume:** checkpoint mid-epoch được seek trực tiếp trong batch sampler. Nếu checkpoint là `next_batch=4000/8140`, DataLoader không đọc lại batch 0–3999 và tqdm bắt đầu ngay khoảng 49%.

**Official code submission:** tạo ZIP có `run.py` và `metadata.yaml` ở root; API chính xác gồm `load_model`, `predict_image`, `predict_batch`; output là 40 probabilities theo `sample["attribute_names"]`; toàn bộ weights/dependencies được đóng gói để chạy không có mạng.

> Lưu ý: `Loading weights...` ở đầu session chỉ là nạp lại SigLIP2 vào GPU/RAM, **không phải train lại**. Model Hub files được cache trên Google Drive để session sau không phải download lại.


In [ ]:
# 1) Mount Drive + persistent Hugging Face cache
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os

CHECKPOINT_DIR = Path("/content/drive/MyDrive/PedestrianAttributeRecognition/Checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/Pedestrian Attribute Recognition/Results")
HF_CACHE_DIR = Path("/content/drive/MyDrive/PedestrianAttributeRecognition/HFCache")

for p in (CHECKPOINT_DIR, RESULTS_DIR, HF_CACHE_DIR):
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")

print("Checkpoint:", CHECKPOINT_DIR)
print("Results   :", RESULTS_DIR)
print("HF cache  :", HF_CACHE_DIR)


In [ ]:
# 2) Clone PAR_UPAR from GitHub — use default branch automatically
REPO_URL = "https://github.com/VoThiKimTrang06101997/PAR_UPAR.git"
REPO_DIR = "/content/PAR_UPAR"

import os, shutil, subprocess

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

result = subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
    text=True,
    capture_output=True,
)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("git clone failed")

os.chdir(REPO_DIR)

print("Branch:", subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip())
print("Commit:", subprocess.check_output(
    ["git", "log", "-1", "--oneline"], text=True
).strip())


## 3) Verify GitHub contains the fixed source

Sau khi tải `PAR_UPAR_Resume_Submission_Fixed_Source.zip`, hãy replace/push các file đó vào repo GitHub của bạn. Cell dưới sẽ **dừng ngay** nếu GitHub vẫn còn bản resume cũ, để tránh bạn tưởng đang dùng fix mới nhưng thực ra Colab clone code cũ.


In [ ]:
# 3) Verify required fixes exist in cloned Git source
from pathlib import Path

required_files = [
    Path("/content/PAR_UPAR/vapor_par/train.py"),
    Path("/content/PAR_UPAR/vapor_par/data.py"),
    Path("/content/PAR_UPAR/vapor_par/checkpoint.py"),
    Path("/content/PAR_UPAR/submission_builder/build_submission.py"),
    Path("/content/PAR_UPAR/submission_builder/submission_runtime/run.py"),
]

for p in required_files:
    if not p.exists():
        raise FileNotFoundError(f"Missing fixed source file: {p}")

train_src = required_files[0].read_text(encoding="utf-8")
data_src = required_files[1].read_text(encoding="utf-8")
run_src = required_files[-1].read_text(encoding="utf-8")

checks = {
    "direct resume batch sampler": "ResumableEpochBatchSampler" in data_src,
    "tqdm starts at saved batch": "initial=epoch_start_batch" in train_src,
    "emergency KeyboardInterrupt checkpoint": "except KeyboardInterrupt" in train_src,
    "official predict_image API": "def predict_image(sample: dict)" in run_src,
    "official predict_batch API": "def predict_batch(samples: list[dict])" in run_src,
    "BATCH_SIZE=64": "BATCH_SIZE = 64" in run_src,
}

for name, ok in checks.items():
    print(f"{name:40s}: {'OK' if ok else 'MISSING'}")

if not all(checks.values()):
    raise RuntimeError(
        "GitHub repo is still the OLD source. Replace it with the fixed source ZIP "
        "provided with this notebook, push to GitHub, then rerun this Colab."
    )


In [ ]:
# 4) Install training dependencies from Git source
%cd /content/PAR_UPAR
%pip install -q --upgrade pip setuptools wheel
%pip install -q --upgrade --upgrade-strategy eager -r requirements.txt

import torch, transformers, huggingface_hub
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)


In [ ]:
# 5) Prepare official UPAR 2027 data; already-downloaded data is reused
%cd /content/PAR_UPAR
!python prepare_data.py


## 6) Inspect the checkpoint BEFORE training

Đây là cell quan trọng để bạn nhìn đúng trạng thái. Với screenshot cũ của bạn, checkpoint có `epoch=0, next_batch_idx=4000, global_step=4000`. Nếu file Drive vẫn chỉ có checkpoint 4000, code **không thể phục hồi 1802 batch sau đó** vì chúng chưa từng được lưu. Từ bản mới, Stop/KeyboardInterrupt sẽ emergency-save và periodic save mặc định mỗi 100 optimizer steps.


In [ ]:
%cd /content/PAR_UPAR
!python inspect_checkpoint.py


## 7) Resume training — do not use `--restart`

`train_colab.py` mặc định AUTO-RESUME. Nó so sánh `resume_step.pt` và `last.pt`, chọn checkpoint tiến xa nhất, restore model/optimizer/scheduler/AMP scaler/RNG, rồi batch sampler seek trực tiếp tới `next_batch_idx`.

Ví dụ đúng sau khi load checkpoint 4000/8140:

```text
[RESUME] ... next_batch=4000/8140 (49.14%), global_step=4000
[RESUME] Direct sampler seek is ENABLED
Train 1/25: 49% 4000/8140 ...
```

Không phải `0/8140`.


In [ ]:
%cd /content/PAR_UPAR

!python train_colab.py \
    --epochs 25 \
    --batch-size 12 \
    --eval-batch-size 24 \
    --num-workers 2 \
    --save-every-steps 100 \
    --log-every-steps 50


In [ ]:
# 8) Inspect checkpoints/logs after interruption or training
%cd /content/PAR_UPAR
!python inspect_checkpoint.py

import pandas as pd
from pathlib import Path

step_log = Path("/content/drive/MyDrive/Pedestrian Attribute Recognition/Results/step_log.csv")
epoch_log = Path("/content/drive/MyDrive/Pedestrian Attribute Recognition/Results/training_log.csv")

if step_log.exists():
    print("\nLast step logs:")
    display(pd.read_csv(step_log).tail(10))

if epoch_log.exists():
    print("\nEpoch logs:")
    display(pd.read_csv(epoch_log).tail(10))


# Build official UPAR 2027 code submission

Challenge runtime is offline. The submission ZIP must contain `run.py` and the **unchanged** official example `metadata.yaml` at archive root. The builder converts `best_calibrated.pt` into a full offline bundle containing the complete SigLIP2 vision encoder, VAPOR-PAR head, semantic prototype buffers, calibration and threshold shift.

The runtime returns **exactly 40 probabilities in `sample["attribute_names"]` order**.


In [ ]:
# 9) Ensure official starter repository exists
import os, shutil, subprocess

OFFICIAL_DIR = "/content/UPAR-Challenge-2027"

if not os.path.exists(OFFICIAL_DIR):
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/speckean/UPAR-Challenge-2027.git",
        OFFICIAL_DIR,
    ])

print("Official starter:", OFFICIAL_DIR)


In [ ]:
# 10) Find participant example metadata.yaml; upload it if Git repo doesn't include it
from pathlib import Path

def find_example_metadata(root):
    root = Path(root)
    scored = []
    if root.exists():
        for p in root.rglob("metadata.yaml"):
            low = str(p).lower()
            if "ingestion_program" in low or "scoring_program" in low:
                continue
            score = 0
            if (p.parent / "run.py").exists():
                score += 100
            if "submission" in low:
                score += 30
            if "example" in low:
                score += 30
            if "starter" in low:
                score += 10
            if score:
                scored.append((score, len(str(p)), p))
    if not scored:
        return None
    scored.sort(key=lambda x: (-x[0], x[1]))
    return scored[0][2]

METADATA_YAML = find_example_metadata(OFFICIAL_DIR)

if METADATA_YAML is None:
    print("Upload metadata.yaml copied unchanged from the official example submission.")
    from google.colab import files
    uploaded = files.upload()
    if "metadata.yaml" not in uploaded:
        raise FileNotFoundError("metadata.yaml is required")
    METADATA_YAML = Path("/content/metadata.yaml")

print("metadata.yaml:", METADATA_YAML)


In [ ]:
# 11) Build offline submission ZIP from best calibrated checkpoint
from pathlib import Path
import subprocess

best_calibrated = Path(
    "/content/drive/MyDrive/PedestrianAttributeRecognition/Checkpoints/best_calibrated.pt"
)
submission_zip = Path(
    "/content/drive/MyDrive/Pedestrian Attribute Recognition/Results/PAR_UPAR_submission.zip"
)

if not best_calibrated.exists():
    raise FileNotFoundError(
        "best_calibrated.pt does not exist yet. Finish training + calibration first."
    )

cmd = [
    "python",
    "/content/PAR_UPAR/submission_builder/build_submission.py",
    "--repo-dir", "/content/PAR_UPAR",
    "--checkpoint", str(best_calibrated),
    "--metadata-yaml", str(METADATA_YAML),
    "--official-starter-dir", OFFICIAL_DIR,
    "--vendor-transformers", "5.2.0",
    "--output", str(submission_zip),
]

print(" ".join(cmd))
subprocess.check_call(cmd)

print("READY TO UPLOAD:", submission_zip)


In [ ]:
# 12) Validate final archive against the documented challenge contract
from pathlib import Path
import zipfile, ast

submission_zip = Path(
    "/content/drive/MyDrive/Pedestrian Attribute Recognition/Results/PAR_UPAR_submission.zip"
)

with zipfile.ZipFile(submission_zip, "r") as zf:
    names = zf.namelist()
    run_text = zf.read("run.py").decode("utf-8")
    metadata_bytes = zf.read("metadata.yaml")

required_root = {
    "run.py",
    "metadata.yaml",
    "model_runtime.py",
    "weights/inference_bundle.pt",
}
missing = required_root - set(names)
assert not missing, f"Missing: {missing}"
assert not any(x.startswith("submission/") for x in names)
assert any(x.startswith("vendor/transformers/") for x in names)

# metadata.yaml must be byte-for-byte unchanged.
assert metadata_bytes == Path(METADATA_YAML).read_bytes()

tree = ast.parse(run_text)
funcs = {n.name for n in tree.body if isinstance(n, ast.FunctionDef)}
assert "load_model" in funcs
assert "predict_image" in funcs
assert "predict_batch" in funcs
assert "BATCH_SIZE = 64" in run_text

print("SUBMISSION VALIDATION: OK")
print("run.py at ZIP root        : YES")
print("metadata.yaml at ZIP root : YES")
print("metadata unchanged        : YES")
print("load_model                : YES")
print("predict_image             : YES")
print("predict_batch             : YES")
print("BATCH_SIZE=64             : YES")
print("offline full weights      : YES")
print("vendored transformers     : YES")
print("ZIP size MiB:", round(submission_zip.stat().st_size / 1024**2, 1))

print("\nFirst archive entries:")
for x in names[:30]:
    print(" ", x)


## Upload

Upload only:

`/content/drive/MyDrive/Pedestrian Attribute Recognition/Results/PAR_UPAR_submission.zip`

under **My Submissions**.

The ZIP contains the **contents** of the submission folder, so `run.py` is at archive root as required.
